# Fruit Classification — 04 · Modelling with PyTorch

PyTorch counterpart to [03_modelling_keras](03_modelling_keras.ipynb): the same two-model comparison on the same data, so the frameworks can be compared directly.

1. a **small custom CNN** trained from scratch,
2. **VGG16 transfer learning** — ImageNet-pretrained, frozen except for a new output layer.

Both models use the **leakage-aware split** from [02_preprocessing](02_preprocessing.ipynb) (`data/splits.csv`); augmentation is applied to the training set only. Training and evaluation run through **shared functions** instead of copy-pasted loops — an earlier version of this notebook duplicated the loops and accidentally appended the VGG16 validation results to the CNN's history lists.

> ⚡ On Colab, switch to a GPU runtime: *Runtime → Change runtime type → GPU*.

## 1. Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir("cnn_vgg16__fruit_classification"):
        !git clone https://github.com/Adriana394/cnn_vgg16__fruit_classification.git
    REPO_DIR = "cnn_vgg16__fruit_classification"
else:
    REPO_DIR = ".."  # this notebook lives in <repo>/notebooks/

DATA_ROOT = os.path.join(REPO_DIR, "data")
SPLITS_CSV = os.path.join(DATA_ROOT, "splits.csv")
print("Data root:", os.path.abspath(DATA_ROOT))

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import VGG16_Weights, vgg16

SEED = 42
IMG_SIZE = 100  # images come in varying sizes (see 01_eda) and are resized here
BATCH_SIZE = 64

torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

## 2. Data

A small `Dataset` reads images straight from the paths in `data/splits.csv`. Both models share the ImageNet normalisation (required for the pretrained VGG16, harmless for the custom CNN).

Training augmentation mirrors the Keras notebook: rotations, crops, flips and colour jitter. `RandomResizedCrop` is limited to `scale=(0.7, 1.0)` — the default lower bound of 0.08 would sometimes train on an 8% sliver of the image.

In [ ]:
splits = pd.read_csv(SPLITS_CSV)
train_df = splits[splits["split"] == "train"].reset_index(drop=True)
val_df = splits[splits["split"] == "val"].reset_index(drop=True)
test_df = splits[splits["split"] == "test"].reset_index(drop=True)

class_names = sorted(splits["label"].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
print(f"Number of classes: {num_classes}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose(
    [
        transforms.RandomRotation(30),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


class FruitDataset(Dataset):
    """Reads images via the filepath/label columns of splits.csv."""

    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(os.path.join(DATA_ROOT, row["filepath"])).convert("RGB")
        return self.transform(image), class_to_idx[row["label"]]


loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=(device == "cuda"))
train_loader = DataLoader(FruitDataset(train_df, train_transform), shuffle=True, **loader_kwargs)
val_loader = DataLoader(FruitDataset(val_df, eval_transform), shuffle=False, **loader_kwargs)
test_loader = DataLoader(FruitDataset(test_df, eval_transform), shuffle=False, **loader_kwargs)

### A look at one augmented training batch

For display the normalisation is undone (`std * x + mean`), otherwise the colours would be distorted.

In [ ]:
def denormalize(tensor):
    """Undo ImageNet normalisation for display; returns an HWC array in [0, 1]."""
    image = tensor.numpy().transpose((1, 2, 0))
    return np.clip(np.array(IMAGENET_STD) * image + np.array(IMAGENET_MEAN), 0, 1)


images, labels = next(iter(train_loader))
grid = torchvision.utils.make_grid(images[:32], nrow=8)

plt.figure(figsize=(14, 7))
plt.imshow(denormalize(grid))
plt.title("Augmented training batch")
plt.axis("off")
plt.show()

## 3. Shared training and evaluation functions

One set of functions for both models:

- `train_model` runs the epoch loop **including validation each epoch**, records history per model, and restores the weights of the best validation epoch (with early stopping),
- `evaluate` computes loss/accuracy on any loader,
- the remaining helpers plot curves, the confusion matrix and example predictions.

This replaces the duplicated loops of the first version, where the VGG16 validation ran only once *after* training and even appended its results to the CNN's history lists.

In [ ]:
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader):
    """Return (mean loss, accuracy) of the model on a data loader."""
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item() * len(labels)
            correct += (outputs.argmax(1) == labels).sum().item()
            count += len(labels)
    return total_loss / count, correct / count


def train_model(model, optimizer, epochs, name, patience=5):
    """Train with per-epoch validation; restore the best-val-loss weights."""
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss, best_state, epochs_without_improvement = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, count = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(1) == labels).sum().item()
            count += len(labels)

        train_loss, train_acc = total_loss / count, correct / count
        val_loss, val_acc = evaluate(model, val_loader)
        for key, value in zip(history, (train_loss, train_acc, val_loss, val_acc)):
            history[key].append(value)
        print(
            f"[{name}] epoch {epoch}/{epochs} — "
            f"train loss {train_loss:.4f}, train acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f}, val acc {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss, best_state, epochs_without_improvement = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"[{name}] early stopping after epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return history


def predict_all(model, loader):
    """Predicted class indices for a (non-shuffled) loader, in order."""
    model.eval()
    predictions = []
    with torch.no_grad():
        for images, _ in loader:
            predictions.append(model(images.to(device)).argmax(1).cpu())
    return torch.cat(predictions).numpy()


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title} — loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title} — accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def show_confusion_and_report(y_true, y_pred, name):
    fig, ax = plt.subplots(figsize=(11, 11))
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, y_pred), display_labels=class_names
    ).plot(ax=ax, xticks_rotation="vertical", colorbar=False, values_format="d")
    ax.set_title(f"{name} — confusion matrix (test set)")
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


def show_sample_predictions(y_pred, name, n=5):
    """Display raw test images with predicted vs. true label.

    Works because the test loader is built with shuffle=False, so
    prediction i corresponds to test_df row i.
    """
    rows = test_df.sample(n=n, random_state=SEED)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    for ax, (i, row) in zip(axes, rows.iterrows()):
        predicted = class_names[y_pred[i]]
        correct = predicted == row["label"]
        ax.imshow(Image.open(os.path.join(DATA_ROOT, row["filepath"])).resize((IMG_SIZE, IMG_SIZE)))
        ax.set_title(f"pred: {predicted}\ntrue: {row['label']}", color="green" if correct else "red", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"{name} — sample test predictions")
    plt.tight_layout()
    plt.show()


y_true = test_df["label"].map(class_to_idx).to_numpy()
results = {}  # collects test metrics for the final comparison

## 4. Small custom CNN

Three convolution blocks (Conv → BatchNorm → ReLU → MaxPool) with increasing filter counts, then a dense classifier with dropout. `MaxPool2d` halves the spatial size after each block (100 → 50 → 25 → 12).

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            conv_block(3, 64),
            conv_block(64, 128),
            conv_block(128, 256),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn_model = CustomCNN(num_classes).to(device)
optimizer_cnn = optim.RMSprop(cnn_model.parameters(), lr=1e-4)
print(cnn_model)

In [ ]:
history_cnn = train_model(cnn_model, optimizer_cnn, epochs=20, name="Custom CNN", patience=5)

In [ ]:
plot_history(history_cnn, "Custom CNN")

### Evaluation on the test set

In [ ]:
cnn_loss, cnn_acc = evaluate(cnn_model, test_loader)
print(f"Custom CNN — test accuracy: {cnn_acc:.4f} | test loss: {cnn_loss:.4f}")
results["Custom CNN"] = {"test accuracy": cnn_acc, "test loss": cnn_loss}

y_pred_cnn = predict_all(cnn_model, test_loader)
show_confusion_and_report(y_true, y_pred_cnn, "Custom CNN")
show_sample_predictions(y_pred_cnn, "Custom CNN")

## 5. VGG16 transfer learning

We load VGG16 with ImageNet weights (`weights=VGG16_Weights.IMAGENET1K_V1` — the old `pretrained=True` is deprecated), **freeze all pretrained parameters** and replace only the final classifier layer with a fresh `Linear` for our 24 classes. Only that new layer is trained — the same frozen-base recipe as in the Keras notebook.

(The first version of this notebook left every parameter trainable, i.e. full fine-tuning from the start — slower and riskier on a small dataset.)

*Optional next step — fine-tuning:* after the new layer has converged, unfreeze the last convolution block (`features[24:]`) and continue with a very low learning rate (e.g. `1e-5`).

In [ ]:
vgg_model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

for param in vgg_model.parameters():
    param.requires_grad = False

# the freshly created layer is trainable by default
vgg_model.classifier[6] = nn.Linear(vgg_model.classifier[6].in_features, num_classes)
vgg_model = vgg_model.to(device)

trainable = sum(p.numel() for p in vgg_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in vgg_model.parameters())
print(f"Trainable parameters: {trainable:,} of {total:,}")

optimizer_vgg = optim.Adam(
    (p for p in vgg_model.parameters() if p.requires_grad), lr=1e-4
)

In [ ]:
history_vgg = train_model(vgg_model, optimizer_vgg, epochs=10, name="VGG16 transfer", patience=3)

In [ ]:
plot_history(history_vgg, "VGG16 transfer")

### Evaluation on the test set

In [ ]:
vgg_loss, vgg_acc = evaluate(vgg_model, test_loader)
print(f"VGG16 transfer — test accuracy: {vgg_acc:.4f} | test loss: {vgg_loss:.4f}")
results["VGG16 transfer"] = {"test accuracy": vgg_acc, "test loss": vgg_loss}

y_pred_vgg = predict_all(vgg_model, test_loader)
show_confusion_and_report(y_true, y_pred_vgg, "VGG16 transfer")
show_sample_predictions(y_pred_vgg, "VGG16 transfer")

## 6. Model comparison

In [ ]:
pd.DataFrame(results).T.round(4)

**How to read these numbers.** As in the Keras notebook, both models are evaluated on the contiguous-block split, so the scores reflect generalisation to unseen views rather than memorisation of near-duplicate frames. They are not comparable to results reported on the official (leaky) Fruits-360 split.

Remaining limitation (see 01_eda): one physical object per class — the test set contains unseen *views*, not unseen *fruits*.

Since notebooks 03 and 04 share the same split, image size, augmentation strategy and evaluation, the comparison table above can be set side by side with the Keras results to compare the two frameworks' implementations of the same experiment.